In [2]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,LongType,TimestampType

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 4, Finished, Available, Finished, False)

 ##### **Create the source configuration table**

In [3]:
source_schema = StructType([
    StructField("source_id", IntegerType(), False),
    StructField("source_file", StringType(), False),
    StructField("bronze_table", StringType(), False),
    StructField("silver_table", StringType(), False),
    StructField("key_column", StringType(), False),
    StructField("load_order", IntegerType(), False),
    StructField("is_active", StringType(), False)
])

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 5, Finished, Available, Finished, False)

In [4]:
source_data = [
    (1, "customers.csv", "bronze_customers",
     "silver_customers", "customer_id", 1, "Y"),
    (2, "products.csv", "bronze_products",
     "silver_products", "product_id", 2, "Y"),
    (3, "orders.csv", "bronze_orders",
     "silver_orders", "order_id", 3, "Y")
]

source_df = spark.createDataFrame(
    source_data,
    schema=source_schema
)

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 6, Finished, Available, Finished, False)

In [5]:
display(source_df)

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cb2d226d-a5b7-49e4-9e94-0ee7f9c5cbda)

In [6]:
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ctl_source_config")
print("ctl_source_config created successfully.")

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 8, Finished, Available, Finished, False)

ctl_source_config created successfully.


##### **Create the pipeline run log table**

In [7]:
log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("rows_written", LongType(), True),
    StructField("status", StringType(), True),
    StructField("log_ts", TimestampType(), True)
])

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 9, Finished, Available, Finished, False)

In [8]:
source_df.write.format("delta").mode("overwrite").saveAsTable("ctl_source_config")
print("ctl_source_config created successfully.")

empty_log_df = spark.createDataFrame([], schema=log_schema)
empty_log_df.write.format("delta").mode("overwrite").saveAsTable("ctl_run_log")

print("ctl_run_log created successfully.")

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 10, Finished, Available, Finished, False)

ctl_source_config created successfully.
ctl_run_log created successfully.


##### **Verify the tables**

In [9]:
print("Source configuration:")
spark.table("ctl_source_config").show(truncate=False)

print("Run log structure:")
spark.table("ctl_run_log").printSchema()

print("Setup completed successfully.")

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 11, Finished, Available, Finished, False)

Source configuration:
+---------+-------------+----------------+----------------+-----------+----------+---------+
|source_id|source_file  |bronze_table    |silver_table    |key_column |load_order|is_active|
+---------+-------------+----------------+----------------+-----------+----------+---------+
|1        |customers.csv|bronze_customers|silver_customers|customer_id|1         |Y        |
|2        |products.csv |bronze_products |silver_products |product_id |2         |Y        |
|3        |orders.csv   |bronze_orders   |silver_orders   |order_id   |3         |Y        |
+---------+-------------+----------------+----------------+-----------+----------+---------+

Run log structure:
root
 |-- run_id: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- table_name: string (nullable = true)
 |-- rows_written: long (nullable = true)
 |-- status: string (nullable = true)
 |-- log_ts: timestamp (nullable = true)

Setup completed successfully.


In [10]:
df = spark.sql("SELECT * FROM LH_Assessment.dbo.ctl_run_log LIMIT 1000")
display(df)

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a6a6fb7a-3752-4674-b269-7829123bb116)

In [11]:
df = spark.sql("SELECT * FROM LH_Assessment.dbo.ctl_source_config LIMIT 1000")
display(df)

StatementMeta(, 8c8043fc-f000-4cb6-8ea2-08b1c5b2ea77, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90b13730-8af6-4f0b-91f2-28e0b24fe951)